# 09 - 二维 Poisson 方程

方程：`u_xx + u_yy = -2*pi^2*sin(pi*x)*sin(pi*y)`。

区域：单位方形 `[0,1] x [0,1]`。

边界：四条边上 `u=0`。

解析解：`u=sin(pi*x)*sin(pi*y)`。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(9)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)

model = MLP().to(device)

def exact_solution(x, y):
    return torch.sin(math.pi * x) * torch.sin(math.pi * y)

In [ ]:
def residual(x, y):
    x.requires_grad_(True)
    y.requires_grad_(True)
    u = model(torch.cat([x, y], dim=1))
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]
    forcing = -2 * math.pi**2 * exact_solution(x, y)
    return u_xx + u_yy - forcing

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(9000):
    x_f = torch.rand(1024, 1, device=device)
    y_f = torch.rand(1024, 1, device=device)
    loss_phys = torch.mean(residual(x_f, y_f)**2)

    s = torch.rand(256, 1, device=device)
    xb = torch.cat([torch.zeros(64,1,device=device), torch.ones(64,1,device=device), s[:64], s[64:128]], dim=0)
    yb = torch.cat([s[128:192], s[192:256], torch.zeros(64,1,device=device), torch.ones(64,1,device=device)], dim=0)
    loss_bc = torch.mean(model(torch.cat([xb, yb], dim=1))**2)

    loss = loss_phys + 20 * loss_bc
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([loss.item(), loss_phys.item(), loss_bc.item()])
    if epoch % 1800 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}")

In [ ]:
grid = 80
xs = torch.linspace(0, 1, grid, device=device)
ys = torch.linspace(0, 1, grid, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], dim=1)
with torch.no_grad():
    pred = model(xy).reshape(grid, grid)
    exact = exact_solution(X, Y)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(exact.cpu(), origin="lower", extent=[0,1,0,1])
plt.title("Exact")
plt.colorbar()
plt.subplot(1, 3, 2)
plt.imshow(pred.cpu(), origin="lower", extent=[0,1,0,1])
plt.title("PINN")
plt.colorbar()
plt.subplot(1, 3, 3)
plt.imshow(torch.abs(pred - exact).cpu(), origin="lower", extent=[0,1,0,1])
plt.title("Abs error")
plt.colorbar()
plt.tight_layout()
plt.show()
print("max abs error=", torch.max(torch.abs(pred - exact)).item())